# The Black Representation in Alpha (`BCEOS <Index>`): End-to-End Quantitative Analysis
**Author:** Tanesha Fuller (NYU) | **Track:** Quantitative Finance & Applied Econometrics  
**Live Terminal:** https://dsc-project-expo2026-tef3573.vercel.app

This reproducible notebook walks sequentially through the complete four-pillar empirical pipeline (`2000-01-03` to `2026-11-24`, `6,765` NYSE trading sessions):
1. **Survivorship-Bias-Free Universe Construction (`src/universe.py`)**: Dynamic point-in-time constituent masking across 40 Black CEO tenures and 36 public tickers (including delisted archives `AET`, `GBT`, `JCP`, `PBPB`, `WBA`) with strict $t+1$ execution lags.
2. **Event-Driven Backtest & Scorecard (`src/backtest.py`, `src/metrics.py`)**: Daily drift accounting, $N_t < 3$ sparse-universe cash buffer ($B_t = \min(1, N_t/3)$), and `10 bps` proportional transaction friction vs. cap-weighted `SPY` and post-May 2003 equal-weighted `RSP`.
3. **Regularized Convex Quadratic Programming (`src/optimizer.py`)**: Shrinkage covariance conditioning and cross-sectional quality tilts solved via `cvxpy` (`CLARABEL` / `OSQP` / `SCS`) under a $35\%$ single-name ceiling.
4. **Factor Attribution, Block Bootstrap & Econometric Time-Series (`src/attribution.py`, `src/bootstrap.py`, `src/timeseries.py`, `src/arima_diagnostics.py`, `src/macro_regimes.py`)**: Fama-French 5-Factor OLS with Newey-West HAC standard errors, Politis-Romano stationary block bootstrap, `GARCH(1,1)` / `GJR-GARCH(1,1)` MLE, walk-forward `ARIMA(1,0,1)`, and six-era macro attribution.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebook':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data'
print(f'Repository root: {ROOT}')
print(f'Data directory exists: {DATA_DIR.exists()}')

## 1. Point-in-Time Universe & Anti-Bias Masking (`src/universe.py`)
We inspect the curated executive roster (`data/black_ceo_roster.csv`), aligned daily price matrix (`data/prices_daily.csv`), and $t+1$ lagged constituent mask (`data/universe_mask.csv`) to verify cross-sectional breadth $N_t$ and survivorship-bias controls.

In [ ]:
roster = pd.read_csv(DATA_DIR / 'black_ceo_roster.csv')
prices = pd.read_csv(DATA_DIR / 'prices_daily.csv', index_col=0, parse_dates=True)
mask = pd.read_csv(DATA_DIR / 'universe_mask.csv', index_col=0, parse_dates=True)

assert len(prices) == len(mask), 'Price matrix and universe mask row counts must match.'
assert (prices.index == mask.index).all(), 'Trading calendar indices must match identically.'

active_nt = mask.sum(axis=1)
print(f'Trading Horizon : {prices.index.min().date()} to {prices.index.max().date()} ({len(prices)} sessions)')
print(f'Roster Entries  : {len(roster)} CEO tenures across {roster["ticker"].nunique()} unique tickers')
print(f'Active Breadth  : Min={int(active_nt.min())}, Mean={active_nt.mean():.2f}, Max={int(active_nt.max())}')
roster.head(10)

## 2. Event-Driven Backtest Scorecard & Benchmark Comparison (`src/backtest.py`, `src/metrics.py`)
We load `data/backtest_results.csv` and `data/summary_metrics.json` to audit cumulative compounding, annualized volatility, Sharpe/Sortino ratios, maximum drawdown, and transaction friction across the full 27-year horizon.

In [ ]:
bt_df = pd.read_csv(DATA_DIR / 'backtest_results.csv', index_col=0, parse_dates=True)
with (DATA_DIR / 'summary_metrics.json').open('r', encoding='utf-8') as f:
    summary_metrics = json.load(f)

print('=== Institutional Summary Metrics JSON ===')
print(json.dumps(summary_metrics, indent=2)[:1800])
bt_df.tail(5)

## 3. Regularized Convex Quadratic Programming (`src/optimizer.py`)
We verify that trailing 63-day shrinkage covariance estimation produces strictly positive semi-definite matrices ($\lambda_{\min} \ge \epsilon > 0$) and that the `cvxpy` QP solver enforces the budget constraint $\sum w_i = B_t$ and the $35\%$ single-name concentration cap.

In [ ]:
from src.optimizer import ConvexPortfolioOptimizer, CovarianceEstimator, condition_covariance_matrix

returns_df = prices[mask.columns].pct_change(fill_method=None).fillna(0.0)
window_rets = returns_df.iloc[-64:-1]
active_vec = mask.iloc[-1].values.astype(bool)

active_cols = mask.columns[active_vec]
cov_raw, shrinkage_intensity = CovarianceEstimator().estimate(window_rets[active_cols])
cov_psd = condition_covariance_matrix(cov_raw)
min_eig = float(np.linalg.eigvalsh(cov_psd).min())

optimizer = ConvexPortfolioOptimizer()
weights_active, objective_val = optimizer.solve(cov_matrix=cov_psd, equity_budget=1.0)
print(f'Ledoit-Wolf Shrinkage Intensity : {shrinkage_intensity:.4f}')
print(f'Minimum Eigenvalue (PSD Check)  : {min_eig:.6e}')
print(f'Active Constituents at T        : {int(active_vec.sum())}')
print(f'Sum of Portfolio Weights        : {weights_active.sum():.6f}')
print(f'Max Single-Issuer Weight        : {weights_active.max():.4%}')

## 4. Fama-French 5-Factor Attribution & Newey-West HAC Inference (`src/attribution.py`)
We regress daily strategy excess returns ($R_{p,t} - R_{f,t}$) on the five Kenneth French daily risk factors (`Mkt-RF`, `SMB`, `HML`, `RMW`, `CMA`) using Newey-West heteroskedasticity- and autocorrelation-consistent (`HAC`) standard errors.

In [ ]:
from src.attribution import (
    FactorAttributionConfig,
    calculate_annualized_alpha_inference,
    download_ff5_daily_factors,
    estimate_ff5_ols_loadings,
)

ff5_df = download_ff5_daily_factors(FactorAttributionConfig())
ew_ols = estimate_ff5_ols_loadings(bt_df['Black_CEO_Net'], ff5_df, scheme_name='BCEOS Equal-Weight (Net)')
alpha_stats = calculate_annualized_alpha_inference(bt_df['Black_CEO_Net'], ff5_df)

print('=== Fama-French 5-Factor HAC Attribution ===')
for k, v in vars(alpha_stats).items():
    print(f'{k:28s}: {v}')

## 5. Econometric Time-Series (`GARCH`, `ARIMA`) & Macro-Regime Diagnostics (`data/timeseries_diagnostics.json`)
Finally, we inspect full-horizon and Pre-/Post-2010 `GARCH(1,1)` and asymmetric `GJR-GARCH(1,1)` maximum-likelihood estimates, out-of-sample walk-forward `ARIMA(1,0,1)` diagnostics, and performance across all six historical macroeconomic regimes.

In [ ]:
with (DATA_DIR / 'timeseries_diagnostics.json').open('r', encoding='utf-8') as f:
    ts_diag = json.load(f)

print('Available diagnostic sections:', list(ts_diag.keys()))
if 'macro_regimes' in ts_diag:
    display(pd.DataFrame(ts_diag['macro_regimes']))
else:
    print(json.dumps(ts_diag, indent=2)[:2000])